In [1]:
import os
import torch
import torch.nn as nn
import numpy as np
from AIce.functions import testloader,redim,getRMSE
from  AIce.models import NNforNorms

device = torch.device(torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else 'cpu')

weightlist=[]
for i in os.listdir('./weights'):
    """"Get the list of all the weights"""
    if i[-3:]=='.pt':
        weightlist.append(i)


data,_,_,means,stds,maxes,_=testloader('../data/DRIFT_DATA_TEST.csv',['u_ERA5','v_ERA5','h_piomas','sic_CDR','x_EASE','y_EASE','bath','sin','cos', 'windnorm']
                                    ,'buoynorm',trainingset_loaded=False, training_file='../data/DRIFT_DATA_TRAIN.csv')# Get the means,stds,maxes using the longest list possible


inputstoweight={}
for k,weight in enumerate(weightlist):
    lst=[]

    with open(f'./weights/{weight[:-3]}.txt', 'r') as f:
        """Gets the inputlist associated with a particular Weight"""
        for i in range(2):
            f.readline()
        splits=f.read().split(':')
        lst=splits[1].strip()
        lst=lst.strip('[]').strip()
        lst=lst.split(',')
        lst=[x.strip()[1:-1] for x in lst]

    mlp256= NNforNorms(len(lst)).to(device) # Creates a model for the specific n_inputs
    mlp256.load_state_dict(torch.load(f'./weights/{weight}', weights_only=True))# loads the weights onto the model
    mlp256.eval()  # if you're doing inference, not continuing training
    name=weight[11:19].strip('_') # I wanted a simpler name to differentiate them

    inputstoweight[name]=lst # saves the list of input into a dict
    specificdf=data[lst] # gets the data for a specific inputlist

    out=mlp256(torch.tensor(specificdf.values, dtype= torch.float32).to(device)).to('cpu')# runs the data into the model 
    data[name]=np.expm1(out.detach().numpy())# save and redimensionalize the model prediction in the datapd with name associated to n_inputs




Bath size is the full test set
Target is buoynorm normalized by log1p
Sin and Cos added
Bathymetry (bath) normalized by maximum
x/y (x_EASE, y_EASE) normalized by maximum
Windnorm normalized by z-score
Wind components (u/v_ERA5) normalized by z-score
Wind components (u/v_ERA5) normalized by z-score


In [2]:
#redimensionalize the inputs

data[['buoynorm','u_ERA5','v_ERA5','h_piomas','sic_CDR','x_EASE','y_EASE','bath','sin','cos', 'windnorm']]= redim(
    data[['buoynorm','u_ERA5','v_ERA5','h_piomas','sic_CDR','x_EASE','y_EASE','bath','sin','cos', 'windnorm']],
    means,
    stds,
    maxes
)

In [3]:
rmse={}

for pred in inputstoweight.keys():
    print(pred)
    rmse[pred]=getRMSE(data['buoynorm'].values,data[f'{pred}'].values)

7inputs
2inputs
4inputs
5inputs
6inputs
9inputs
10inputs
3inputs


In [4]:
print(rmse)

{'7inputs': np.float64(8901237692442176.0), '2inputs': np.float64(6.495467614921789), '4inputs': np.float64(8.445659896182164), '5inputs': np.float64(8.406930460762975), '6inputs': np.float64(57.77208034283163), '9inputs': np.float64(2230.7094795718253), '10inputs': np.float64(2211.994081211778), '3inputs': np.float64(5.854516949675476)}
